In [ ]:
# Bootstrap: copy the nanowm-code dataset into /kaggle/working and cd into it.
import os, shutil, subprocess, sys, zipfile
from pathlib import Path

inputs = Path("/kaggle/input")
roots = list((inputs / "nanowm-code").rglob("pyproject.toml"))
if not roots:
    roots = list(inputs.rglob("pyproject.toml"))
if len(roots) != 1:
    raise RuntimeError(f"Expected exactly one project root, found {len(roots)}: {roots}")
mounted = roots[0].parent
root = Path("/kaggle/working/project")
root.mkdir(parents=True, exist_ok=True)
shutil.copytree(mounted, root, dirs_exist_ok=True)
for archive in mounted.glob("*.zip"):
    with zipfile.ZipFile(archive) as handle:
        handle.extractall(root)
os.chdir(root)
sys.path.insert(0, str(root))
print("project root:", root)
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

In [ ]:
# HARD GATE. machine_shape only *requests* 2xT4; nothing guarantees it.
# Numbers measured on a P100 would look plausible and be worthless, and
# would have cost weekly quota to produce. Stop here instead.
subprocess.run([sys.executable, "scripts/preflight_gpu.py"], check=True)

In [ ]:
# Confirm the code behaves the same on Kaggle as it does locally before
# trusting any measurement taken here. Cheap, and catches environment skew.
subprocess.run([sys.executable, "-m", "pytest", "tests/", "-q"], check=True)

In [ ]:
# The approved moderngl/EGL probe: decides whether the renderer's primary
# backend works on Kaggle or whether we ship the numpy fallback as primary.
# Non-fatal by design -- a FAIL here is a finding, not a reason to stop.
print(subprocess.run(
    [sys.executable, "scripts/preprocess/check_kaggle_gl.py"],
    capture_output=True, text=True,
).stdout)

In [ ]:
# The measurement M0.5 has been missing. Single device: DDP scaling is a
# separate question and multiplying by world_size is the wrong way to answer
# it. Batch 8 rather than 2 so kernel-launch overhead does not dominate the
# way it does at toy batch sizes.
subprocess.run([
    sys.executable, "scripts/profile_dit.py",
    "--presets", "5m", "15m", "40m",
    "--tokens", "16", "64", "256",
    "--batch-size", "8",
    "--steps", "20",
    "--warmup", "3",
    "--time-budget", "45",
    "--out", "/kaggle/working/profile_2xt4.json",
], check=True)

In [ ]:
import json
data = json.load(open("/kaggle/working/profile_2xt4.json"))
print(data["device"], "x", data["device_count"], "| torch", data["torch"])
hdr = f"{'preset':>6} {'tok/f':>6} {'seq':>6} {'compiled':>9} {'step_ms':>9} {'VRAM_GB':>8} {'MFU%':>6} {'steps/GPUh':>11}"
print(hdr); print("-" * len(hdr))
for r in data["results"]:
    if r.get("error") or r.get("oom") or "step_time_median_s" not in r:
        print(f"{r.get('preset','?'):>6} {r.get('tokens_per_frame','?'):>6} "
              f"{'':>6} {'':>9}  {'OOM' if r.get('oom') else 'ERROR'}")
        continue
    print(f"{r['preset']:>6} {r['tokens_per_frame']:>6} {r['sequence_length']:>6} "
          f"{str(r['compiled']):>9} {r['step_time_median_s']*1000:>9.1f} "
          f"{r['peak_vram_gb']:>8.2f} {r['mfu_percent']:>6.2f} {3600/r['step_time_median_s']:>11,.0f}")